# Highly Optimized Docling PDF Ingestion Benchmark
This notebook runs an incredibly efficient, highly parallelized Docling ingestion pipeline. 
It processes PDFs from a source directory, saves the structured output as JSON to a destination directory, and meticulously tracks latency, throughput, and error metrics.

**Why this is optimal:**
1. Uses `ProcessPoolExecutor` to bypass the Python GIL and achieve maximum CPU utilization.
2. Initializes the Docling models only once per worker process (to avoid cold-start overhead).
3. Outputs parsed documents straight to disk for downstream tasks (like embedding).


In [9]:
import os
import time
import json
import psutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- CRITICAL OPTIMIZATION: Thread Thrashing Prevention ---
# When running ProcessPoolExecutor with heavily optimized C++ libraries (PyTorch, ONNX), 
# they will attempt to use all CPU cores. If 8 processes each try to use 8 threads, 
# you get extreme context-switching overhead (thrashing). We lock them to 1 thread per process.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Docling imports
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

# Configuration
SOURCE_DIR = Path("../nlp_ml_gcs_pdfs")  # Path to your PDFs
OUTPUT_DIR = Path("../data/docling_parsed_output")
# MAX_WORKERS = max(1, psutil.cpu_count(logical=False) - 1)  # Leave 1 physical core for OS
MAX_WORKERS = 1
MAX_FILES = 1000 # Number of files to process for this benchmark

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using {MAX_WORKERS} parallel workers.")


Using 1 parallel workers.


## 1. Defining the Worker Function
Because we are using multiprocessing, we must handle initialization carefully. Loading ML models inside the main thread and passing them to workers is inefficient and often crashes. Instead, we initialize the `DocumentConverter` globally **inside each worker process**.


In [10]:
# --- CRITICAL JUPYTER FIX ---
# Multiprocessing on macOS uses 'spawn', which crashes if worker functions 
# are defined directly inside a Jupyter Notebook. 
# We must import them from an external .py file.

from ingestion_worker import init_worker, process_single_pdf


## 2. Orchestrating the Parallel Pipeline
We will now use `ProcessPoolExecutor` to map the PDFs to our workers.


In [ ]:
def run_ingestion_pipeline(source_dir: Path, output_dir: Path, max_files: int = 100, max_workers: int = 4):
    pdf_files = list(source_dir.glob("*.pdf"))[:max_files]
    print(f"Found {len(pdf_files)} PDFs to process.")
    
    metrics = []
    start_total = time.perf_counter()
    
    # Use ProcessPoolExecutor to max out CPU cores
    with ProcessPoolExecutor(max_workers=max_workers, initializer=init_worker) as executor:
        # Submit all tasks
        futures = {executor.submit(process_single_pdf, pdf, output_dir): pdf for pdf in pdf_files}
        
        # Stream results as they complete
        for future in tqdm(as_completed(futures), total=len(futures), desc="Parsing PDFs"):
            metrics.append(future.result())
            
    end_total = time.perf_counter()
    
    df_metrics = pd.DataFrame(metrics)
    total_time = end_total - start_total
    print(f"\nPipeline completed in {total_time:.2f} seconds.")
    
    return df_metrics, total_time

# Run the pipeline!
df_metrics, total_time = run_ingestion_pipeline(
    source_dir=SOURCE_DIR, 
    output_dir=OUTPUT_DIR, 
    max_files=MAX_FILES, 
    max_workers=MAX_WORKERS
)


Found 700 PDFs to process.


/Users/tri/.local/share/uv/python/cpython-3.10.19-macos-aarch64-none/lib/python3.10/multiprocessing/resource_tracker.py:104: UserWarning: resource_tracker: process died unexpectedly, relaunching.  Some resources might leak.
  warnings.warn('resource_tracker: process died unexpectedly, '
Parsing PDFs:   9%|▉         | 65/700 [17:50<2:54:17, 16.47s/it]


## 3. Analytics and Efficiency Metrics
Let's analyze how well the ingestion performed.


In [ ]:
# Summary Statistics
success_rate = (df_metrics['status'] == 'SUCCESS').mean() * 100
avg_time = df_metrics.loc[df_metrics['status'] == 'SUCCESS', 'processing_time_sec'].mean()
throughput = len(df_metrics) / total_time

print(f"=== INGESTION SUMMARY ===")
print(f"Total PDFs Processed: {len(df_metrics)}")
print(f"Success Rate:         {success_rate:.2f}%")
print(f"Total Wall Time:      {total_time:.2f} s")
print(f"Throughput:           {throughput:.2f} PDFs / sec")
print(f"Avg Time per PDF:     {avg_time:.2f} s")

# Display failures if any
failures = df_metrics[df_metrics['status'] == 'FAILED']
if not failures.empty:
    print(f"\nEncountered {len(failures)} failures:")
    display(failures[['filename', 'error_msg']].head())

# Plotting the distribution of processing times
plt.figure(figsize=(10, 5))
sns.histplot(data=df_metrics[df_metrics['status'] == 'SUCCESS'], x='processing_time_sec', bins=30, kde=True)
plt.title('Distribution of Docling Processing Times per PDF')
plt.xlabel('Processing Time (seconds)')
plt.ylabel('Count')
plt.show()

# Relationship between File Size and Processing Time
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_metrics[df_metrics['status'] == 'SUCCESS'], x='file_size_kb', y='processing_time_sec', alpha=0.6)
plt.title('File Size vs Processing Time')
plt.xlabel('File Size (KB)')
plt.ylabel('Processing Time (seconds)')
plt.show()
